# Kaggle Predict F1 Pit Stops: Model Optimization and Ensemble

A focused workflow for LightGBM tuning, feature-set validation, blend checks, calibration diagnostics, and final submission generation.


## 1. Setup


In [ ]:
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10, "legend.frameon": False})

RANDOM_STATE = 42
TARGET = "PitNextLap"
ID_COL = "id"

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler


def evaluate_predictions(y_true, y_pred) -> dict:
    y_pred = np.clip(y_pred, 1e-6, 1 - 1e-6)
    return {
        "roc_auc": roc_auc_score(y_true, y_pred),
        "average_precision": average_precision_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_pred),
    }


def get_feature_columns(df: pd.DataFrame, target: str = TARGET, id_col: str = ID_COL):
    features = df.drop(columns=[c for c in [target, id_col] if c in df.columns])
    cat_cols = features.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in features.columns if c not in cat_cols]
    return num_cols, cat_cols


def make_tree_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    num_cols, cat_cols = get_feature_columns(df)
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ], remainder="drop")

RUN_FAST = True
FAST_SAMPLE_SIZE = 180_000
N_SPLITS = 5
SELECTED_FEATURE_SET = "safe_plus_ratios_no_driver"


## 2. Load Data


In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e5"),
        Path("/kaggle/input/playground-series-s6e5"),
        Path("../input/competitions/playground-series-s6e5"),
        Path("../input/playground-series-s6e5"),
        Path("data"),
        Path("../data"),
        Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find train.csv and test.csv. Update DATA_DIR manually.")


def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5:
                out[col] = out[col].astype("category")
    return out


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = reduce_memory_usage(pd.read_csv(DATA_DIR / "train.csv"))
test = reduce_memory_usage(pd.read_csv(DATA_DIR / "test.csv"))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"DATA_DIR: {DATA_DIR}")
print(f"train: {train.shape}")
print(f"test: {test.shape}")
print(f"target positive rate: {train[TARGET].mean():.5f}")
train.head()

if RUN_FAST and len(train) > FAST_SAMPLE_SIZE:
    train_eval = (train.groupby(TARGET, group_keys=False)
        .apply(lambda x: x.sample(frac=min(1.0, FAST_SAMPLE_SIZE / len(train)), random_state=RANDOM_STATE))
        .sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True))
else:
    train_eval = train.copy()
print("train_eval:", train_eval.shape)


## 3. Feature Sets


In [ ]:
def add_features(df: pd.DataFrame, include_safe: bool = True, include_ratios: bool = True) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    if include_safe and {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedRaceLaps"] = out["LapNumber"] / out["RaceProgress"].clip(lower=eps)
        out["EstimatedLapsRemaining"] = out["EstimatedRaceLaps"] - out["LapNumber"]
        out["LapNumber_x_RaceProgress"] = out["LapNumber"] * out["RaceProgress"]

    if include_ratios and {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreLife_to_LapNumber"] = out["TyreLife"] / out["LapNumber"].clip(lower=eps)

    if include_ratios and {"TyreLife", "EstimatedRaceLaps"}.issubset(out.columns):
        out["TyreLife_to_EstimatedRaceLaps"] = out["TyreLife"] / out["EstimatedRaceLaps"].clip(lower=eps)

    if include_safe and {"LapTime (s)", "LapTime_Delta"}.issubset(out.columns):
        out["LapTime_plus_Delta"] = out["LapTime (s)"] + out["LapTime_Delta"]
        out["AbsLapTime_Delta"] = out["LapTime_Delta"].abs()

    if include_safe and {"Position", "Position_Change"}.issubset(out.columns):
        out["PreviousPositionApprox"] = out["Position"] - out["Position_Change"]
        out["AbsPosition_Change"] = out["Position_Change"].abs()

    if include_safe and "Compound" in out.columns:
        compound = out["Compound"].astype(str)
        out["IsSoft"] = (compound == "SOFT").astype("int8")
        out["IsMedium"] = (compound == "MEDIUM").astype("int8")
        out["IsHard"] = (compound == "HARD").astype("int8")
        out["IsWetOrIntermediate"] = compound.isin(["WET", "INTERMEDIATE"]).astype("int8")

    return reduce_memory_usage(out)

feature_set_configs = {
    "raw": {"include_safe": False, "include_ratios": False, "drop_cols": []},
    "safe_engineered": {"include_safe": True, "include_ratios": False, "drop_cols": []},
    "safe_plus_ratios": {"include_safe": True, "include_ratios": True, "drop_cols": []},
    "safe_plus_ratios_no_driver": {"include_safe": True, "include_ratios": True, "drop_cols": ["Driver"]},
    "safe_plus_ratios_no_pitstop": {"include_safe": True, "include_ratios": True, "drop_cols": ["PitStop"]},
}

def prepare_feature_set(df: pd.DataFrame, feature_set: str) -> pd.DataFrame:
    config = feature_set_configs[feature_set]
    out = add_features(df, include_safe=config["include_safe"], include_ratios=config["include_ratios"])
    return out.drop(columns=[c for c in config.get("drop_cols", []) if c in out.columns])

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)


## 4. LightGBM Tuning


In [ ]:
base_lgbm_params = {
    "objective": "binary",
    "n_estimators": 900 if RUN_FAST else 1800,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
}

param_grid = {
    "learning_rate": [0.015, 0.02, 0.03, 0.04],
    "num_leaves": [63, 95, 127],
    "min_child_samples": [60, 100, 150],
    "subsample": [0.85, 0.95, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "reg_alpha": [0.03, 0.1, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
    "max_bin": [127, 255],
}
N_ITER = 8 if RUN_FAST else 25
candidate_params = list(ParameterSampler(param_grid, n_iter=N_ITER, random_state=RANDOM_STATE))

def cv_lgbm_params(params: dict, feature_set: str = SELECTED_FEATURE_SET) -> dict:
    data = prepare_feature_set(train_eval, feature_set)
    X = data.drop(columns=[TARGET])
    y = data[TARGET].astype("int8")
    oof = np.zeros(len(X), dtype=np.float32)
    start = time.time()
    for train_idx, valid_idx in cv.split(X, y):
        X_train_raw, X_valid_raw = X.iloc[train_idx], X.iloc[valid_idx]
        y_train = y.iloc[train_idx]
        preprocessor = make_tree_preprocessor(X_train_raw)
        X_train = preprocessor.fit_transform(X_train_raw)
        X_valid = preprocessor.transform(X_valid_raw)
        model = LGBMClassifier(**base_lgbm_params, **params)
        model.fit(X_train, y_train)
        oof[valid_idx] = model.predict_proba(X_valid)[:, 1]
        del preprocessor, X_train, X_valid, model
        gc.collect()
    scores = evaluate_predictions(y, oof)
    scores.update(params)
    scores["fit_seconds"] = time.time() - start
    return scores

tuning_results = pd.DataFrame([cv_lgbm_params(params) for params in candidate_params]).sort_values("roc_auc", ascending=False).reset_index(drop=True)
tuning_results.head(10)


## 5. Feature Validation


In [ ]:
INT_LGBM_PARAMS = {"num_leaves", "min_child_samples", "max_bin", "n_estimators", "max_depth"}

def clean_lgbm_params(params: dict) -> dict:
    cleaned = {}
    for key, value in params.items():
        if key in INT_LGBM_PARAMS and pd.notna(value):
            cleaned[key] = int(value)
        elif isinstance(value, (np.floating, float)):
            cleaned[key] = float(value)
        else:
            cleaned[key] = value
    return cleaned

best_lgbm_params = clean_lgbm_params({key: tuning_results.iloc[0][key] for key in param_grid.keys()})
print(best_lgbm_params)

def cv_feature_set(feature_set: str) -> dict:
    data = prepare_feature_set(train_eval, feature_set)
    X = data.drop(columns=[TARGET])
    y = data[TARGET].astype("int8")
    oof = np.zeros(len(X), dtype=np.float32)
    for train_idx, valid_idx in cv.split(X, y):
        X_train_raw, X_valid_raw = X.iloc[train_idx], X.iloc[valid_idx]
        y_train = y.iloc[train_idx]
        preprocessor = make_tree_preprocessor(X_train_raw)
        X_train = preprocessor.fit_transform(X_train_raw)
        X_valid = preprocessor.transform(X_valid_raw)
        model = LGBMClassifier(**base_lgbm_params, **best_lgbm_params)
        model.fit(X_train, y_train)
        oof[valid_idx] = model.predict_proba(X_valid)[:, 1]
        del preprocessor, X_train, X_valid, model
        gc.collect()
    scores = evaluate_predictions(y, oof)
    scores.update({"feature_set": feature_set, "n_features": X.shape[1]})
    return scores

feature_results = pd.DataFrame([cv_feature_set(fs) for fs in feature_set_configs]).sort_values("roc_auc", ascending=False).reset_index(drop=True)
feature_results


## 6. LightGBM + XGBoost Blend


In [ ]:
selected_feature_set = feature_results.iloc[0]["feature_set"]
data = prepare_feature_set(train_eval, selected_feature_set)
X = data.drop(columns=[TARGET])
y = data[TARGET].astype("int8")

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 700 if RUN_FAST else 1400,
    "learning_rate": 0.025,
    "max_depth": 7,
    "min_child_weight": 8,
    "subsample": 0.85,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.1,
    "reg_lambda": 3.0,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

def cv_model(factory):
    oof = np.zeros(len(X), dtype=np.float32)
    for train_idx, valid_idx in cv.split(X, y):
        X_train_raw, X_valid_raw = X.iloc[train_idx], X.iloc[valid_idx]
        y_train = y.iloc[train_idx]
        preprocessor = make_tree_preprocessor(X_train_raw)
        X_train = preprocessor.fit_transform(X_train_raw)
        X_valid = preprocessor.transform(X_valid_raw)
        model = factory()
        model.fit(X_train, y_train)
        oof[valid_idx] = model.predict_proba(X_valid)[:, 1]
        del preprocessor, X_train, X_valid, model
        gc.collect()
    return oof

lgbm_oof = cv_model(lambda: LGBMClassifier(**base_lgbm_params, **best_lgbm_params))
xgb_oof = cv_model(lambda: XGBClassifier(**xgb_params))
blend_rows = []
for w in np.linspace(0, 1, 21):
    pred = w * lgbm_oof + (1 - w) * xgb_oof
    scores = evaluate_predictions(y, pred)
    scores.update({"lgbm_weight": w, "xgb_weight": 1 - w})
    blend_rows.append(scores)
blend_results = pd.DataFrame(blend_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
blend_results.head(10)


## 7. Calibration and Error Analysis


In [ ]:
best_blend = blend_results.iloc[0]
best_oof = best_blend["lgbm_weight"] * lgbm_oof + best_blend["xgb_weight"] * xgb_oof

oof_diagnostics = X.copy()
oof_diagnostics["target"] = y.to_numpy()
oof_diagnostics["prediction"] = np.clip(best_oof, 1e-6, 1 - 1e-6)
oof_diagnostics["prediction_bin"] = pd.qcut(oof_diagnostics["prediction"], q=10, duplicates="drop")

calibration = (
    oof_diagnostics.groupby("prediction_bin", observed=True)
    .agg(
        rows=("target", "size"),
        predicted_rate=("prediction", "mean"),
        actual_rate=("target", "mean"),
    )
    .reset_index()
)
calibration["calibration_gap"] = calibration["predicted_rate"] - calibration["actual_rate"]
display(calibration)

fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(data=calibration, x="predicted_rate", y="actual_rate", marker="o", color=sns.color_palette("viridis", 8)[5], ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="0.45", linewidth=1)
ax.set_title("OOF calibration by prediction decile")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed pit-next-lap rate")
plt.show()

threshold_rows = []
positive_total = max(oof_diagnostics["target"].sum(), 1)
for top_pct in [0.5, 1, 2, 5, 10, 20]:
    cutoff = np.percentile(oof_diagnostics["prediction"], 100 - top_pct)
    selected = oof_diagnostics[oof_diagnostics["prediction"] >= cutoff]
    threshold_rows.append({
        "top_pct": top_pct,
        "threshold": cutoff,
        "selected_rows": len(selected),
        "precision": selected["target"].mean(),
        "recall": selected["target"].sum() / positive_total,
    })
threshold_summary = pd.DataFrame(threshold_rows)
display(threshold_summary)

def slice_summary(column: str, min_rows: int = 1000) -> pd.DataFrame:
    rows = []
    grouped = oof_diagnostics.groupby(column, observed=True)
    for value, part in grouped:
        if len(part) < min_rows:
            continue
        row = {
            "slice": column,
            "value": value,
            "rows": len(part),
            "actual_rate": part["target"].mean(),
            "predicted_rate": part["prediction"].mean(),
            "calibration_gap": part["prediction"].mean() - part["target"].mean(),
        }
        if part["target"].nunique() == 2:
            row["roc_auc"] = roc_auc_score(part["target"], part["prediction"])
        rows.append(row)
    return pd.DataFrame(rows)

oof_diagnostics["race_progress_band"] = pd.cut(
    oof_diagnostics["RaceProgress"],
    bins=[0, 0.15, 0.30, 0.50, 0.75, 1.01],
    labels=["early", "early_mid", "mid", "late", "final"],
    include_lowest=True,
)

slice_tables = []
for column in ["Compound", "Stint", "Race", "race_progress_band"]:
    if column in oof_diagnostics.columns:
        slice_tables.append(slice_summary(column))

slice_diagnostics = pd.concat(slice_tables, ignore_index=True).sort_values("calibration_gap", key=lambda s: s.abs(), ascending=False)
display(slice_diagnostics.head(30))


## 8. Train Final Submission


In [ ]:
test_selected = prepare_feature_set(test, selected_feature_set)
train_full_selected = prepare_feature_set(train, selected_feature_set)
X_full = train_full_selected.drop(columns=[TARGET])
y_full = train_full_selected[TARGET].astype("int8")

preprocessor = make_tree_preprocessor(X_full)
X_train_processed = preprocessor.fit_transform(X_full)
X_test_processed = preprocessor.transform(test_selected)

best_blend = blend_results.iloc[0]
final_lgbm = LGBMClassifier(**base_lgbm_params, **best_lgbm_params)
final_lgbm.fit(X_train_processed, y_full)
pred = best_blend["lgbm_weight"] * final_lgbm.predict_proba(X_test_processed)[:, 1]

if best_blend["xgb_weight"] > 0:
    final_xgb = XGBClassifier(**xgb_params)
    final_xgb.fit(X_train_processed, y_full)
    pred += best_blend["xgb_weight"] * final_xgb.predict_proba(X_test_processed)[:, 1]

submission = sample_submission.copy()
submission[TARGET] = np.clip(pred, 1e-6, 1 - 1e-6)
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print("Selected feature set:", selected_feature_set)
print("Best blend:", best_blend.to_dict())
submission.head()
